In [2]:
import json, torch
from pathlib import Path
from transformers import pipeline, AutoModelForSpeechSeq2Seq, AutoProcessor

d:\MSc\3. Spring 2026\CSE715\Project\vae-audio-clustering\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
DATASET_DIR = Path(r"D:\MSc\3. Spring 2026\CSE715\Project\shorter_wavs3sec")      # root folder with genre subdirectories
OUTPUT_DIR  = Path("./extracted_lyrics_small")       # where transcripts will be saved
AUDIO_EXTS  = {".mp3", ".wav", ".flac", ".ogg", ".m4a"}
LANGUAGE    = "bengali"               # force Bengali decoding
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
DATASET_DIR.exists()

True

In [5]:
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

In [7]:
if DEVICE == "cuda":
    start_vram = torch.cuda.memory_allocated() / 1e9
    print(f"Initial GPU Memory: {start_vram:.2f} GB")

Initial GPU Memory: 0.00 GB


In [8]:
model_id="ashrafulparan/whisper-small-bangla"

In [9]:
print(f"Loading {model_id} on {DEVICE}...")

Loading ashrafulparan/whisper-small-bangla on cuda...


In [14]:
pipe = pipeline(
    "automatic-speech-recognition",
    model=model_id,
    dtype=torch.float32,
    generate_kwargs={"language": LANGUAGE, "task": "transcribe"},
)

Loading weights: 100%|██████████| 479/479 [00:00<00:00, 9066.70it/s]


In [15]:
# After loading the pipeline:
print(pipe.model.device)   # should print: cuda:0

cuda:0


In [12]:
def transcribe_dataset(pipe, dataset_dir, output_dir):
    dataset_path = Path(dataset_dir)
    output_path  = Path(output_dir)
    results = {}

    for genre_dir in sorted(dataset_path.iterdir()):
        if not genre_dir.is_dir():
            continue

        genre = genre_dir.name
        results[genre] = {}
        genre_output = output_path / genre
        genre_output.mkdir(parents=True, exist_ok=True)

        audio_files = [
            f for f in sorted(genre_dir.iterdir())
            if f.suffix.lower() in AUDIO_EXTS
        ]

        print(f"\n🎵 Genre: {genre} — {len(audio_files)} files")

        for audio_file in audio_files:
            print(f"  Transcribing: {audio_file.name} ...", end=" ", flush=True)
            try:
                output = pipe(str(audio_file))
                transcript = output["text"].strip()

                # Save individual .txt file
                txt_path = genre_output / (audio_file.stem + ".txt")
                txt_path.write_text(transcript, encoding="utf-8")

                results[genre][audio_file.name] = transcript
                print("✓")

            except Exception as e:
                print(f"✗ ERROR: {e}")
                results[genre][audio_file.name] = f"ERROR: {e}"

    # Save combined JSON
    json_path = output_path / "all_lyrics.json"
    json_path.write_text(
        json.dumps(results, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )
    print(f"\n✅ Done! Results saved to: {output_path}")
    return results


In [ ]:
transcribe_dataset(pipe, DATASET_DIR, OUTPUT_DIR)

### Testing transcription accuracy with our dataset

In [17]:
SOURCE_PATH = Path(r"D:\MSc\3. Spring 2026\CSE715\Project\yt autocap dataset")
AUDIO_DIR = SOURCE_PATH / "audio"
METADATA_CSV = SOURCE_PATH / "transcripts.csv"

In [18]:
# ── GPU check ─────────────────────────────────────────────────────────────────
assert torch.cuda.is_available(), "CUDA not available!"
print(f"GPU: {torch.cuda.get_device_name(0)}")

GPU: NVIDIA GeForce RTX 2050


In [19]:
print(f"Model on: {pipe.model.device}")

Model on: cuda:0


In [20]:
import pandas as pd

In [21]:
# ── Load metadata ─────────────────────────────────────────────────────────────
df = pd.read_csv(METADATA_CSV)
# df has columns: id, sentence
# expecting audio files at AUDIO_DIR / (id + AUDIO_EXT)

print(f"\nTotal samples: {len(df)}")


Total samples: 341


In [22]:
from jiwer import wer, cer

In [26]:
# ── Run evaluation ────────────────────────────────────────────────────────────
records = []
skipped = 0

for i, row in df.iterrows():
    if i == 80:
        break
    audio_path = Path(AUDIO_DIR) / (row["id"] + ".wav")

    if not audio_path.exists():
        print(f"  [MISSING] {audio_path.name}")
        skipped += 1
        continue

    reference = str(row["sentence"]).strip()

    try:
        result     = pipe(str(audio_path), generate_kwargs={"language": "bengali", "task": "transcribe"})
        hypothesis = result["text"].strip()

        sample_wer = wer(reference, hypothesis)
        sample_cer = cer(reference, hypothesis)

        records.append({
            "id":         row["id"],
            "reference":  reference,
            "hypothesis": hypothesis,
            "wer":        round(sample_wer, 4),
            "cer":        round(sample_cer, 4),
        })

        print(f"[{i+1}/{len(df)}] {row['id']}")
        print(f"  REF : {reference}")
        print(f"  HYP : {hypothesis}")
        print(f"  WER : {sample_wer:.2%}  |  CER : {sample_cer:.2%}")

    except Exception as e:
        print(f"  [ERROR] {row['id']}: {e}")
        skipped += 1

[1/341] 8dLOs8LK4OQ_seg_00001
  REF : না না না না না স্বপ্নের বালুকায়
  HYP : না না না না না না না না না না না না না না না না না না না না না না না স্বপ্নের বালু খায়
  WER : 285.71%  |  CER : 181.25%
[2/341] 8dLOs8LK4OQ_seg_00002
  REF : কেউ কি পালুকায় যদি না আসে ভেজা দিন ইচ্ছে হিমালয় হয় না কভু ক্ষয় হৃদয়ে
  HYP : কেউ কী পালুকায় যদি না সে ভেজারিং, ইচ্ছের হেই মাল অয় অয় না, কপুক অয় হৃদয়
  WER : 86.67%  |  CER : 38.36%
[3/341] 8dLOs8LK4OQ_seg_00003
  REF : থাকে অমলিন কিছু স্বপ্ন কিছু ইচ্ছে এই আমায় টেনে নিচ্ছে তোমার কাছে বারবার
  HYP : থাকে অমলিম, কিছু স্বপ্ন কিছু ইচ্ছে, হেই আমায় তেইলে নিচ্ছে তোমার কাছে বারে বা
  WER : 53.85%  |  CER : 18.06%
[4/341] 8dLOs8LK4OQ_seg_00004
  REF : কেউ না জানুক আমি তো জানি আমি তোমার কেউ না জানুক তুমি তো জানো তুমি আমার
  HYP : কেউরা জানি, আমি তো জানি, আমি তো মা। কেউরা জানু, তুমি তো জানু, তুমি আমা।
  WER : 62.50%  |  CER : 18.57%
[5/341] 8dLOs8LK4OQ_seg_00005
  REF : কেউ না জানো আমি তো জানি আমি তোমার
  HYP : কেউরা জানি, আমি তো জানি, আমি তো মা
  WER : 75.0

In [27]:
skipped

0

In [28]:
results_df = pd.DataFrame(records)
# results_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")  # utf-8-sig for Excel compatibility

In [29]:
if len(records) > 0:
    avg_wer = results_df["wer"].mean()
    avg_cer = results_df["cer"].mean()

    print("\n" + "="*50)
    print(f"  Evaluated : {len(records)} samples")
    print(f"  Skipped   : {skipped} samples")
    print(f"  Avg WER   : {avg_wer:.2%}  (lower is better)")
    print(f"  Avg CER   : {avg_cer:.2%}  (lower is better)")
    print(f"  Accuracy  : {(1 - avg_wer):.2%}  (word-level)")
    print("="*50)
else:
    print("\nNo samples were evaluated successfully.")


  Evaluated : 80 samples
  Skipped   : 0 samples
  Avg WER   : 78.00%  (lower is better)
  Avg CER   : 39.75%  (lower is better)
  Accuracy  : 22.00%  (word-level)
